# 032 — GM record selection, AvgSA([0, 3]) — Stage 1 (compute)

The slow selection step. Runs the configurable multi-round selection + optimisation engine
(`build_final_ensembles`) to build the final per-`(site, iml)` record ensembles conditioned on
**AvgSA([0, 3])**, using the shared `SELECTION_CONFIG` (4 rounds dropping progressively more
causal-parameter bounds; round 4 is a shuffled-DB best-of retry).

**Incremental** — like nb `031`, this notebook calls `find_stale_stripes` and only (re)selects the
stripes that are missing or stale on disk. The stale check runs before any disagg data is read, and
only the shards of the sites that turn out to be stale are then loaded — an all-valid run opens none
of them. Per-stripe fingerprints name a single site's shard; the per-round stage caches and
`final_ensembles` cover a whole batch, so they fingerprint the digest of the entire shard set (the batch nb `031` just built gcim for). The round loop is
per-`(site, iml)` independent, so selecting a subset is equivalent to selecting those keys within the
full set. The batch is then **merged into `AvgSA_03_final_ensembles.pickle`** so that file stays the
COMPLETE `(site, iml)` set (it is read as complete by `summarise_record_availability.py` and
provenance-stamped by nb `040`). The canonical per-stripe record is written by nb `033`; the
per-round stage caches here are per-batch scratch. Set `FORCE_RECOMPUTE = True` to reselect every
wanted stripe.

**Excluded records** (updated 2026-10-29). The records known to be unusable are removed from the
database itself by nb 030 (listed in `unavailable_records_ngasub.csv`), so they cannot be selected.
The in-memory filter (`gm_db_for_selection`, cell "Set up + filter the database") stays, for a
record found bad after a selection (`setup_AvgSA03_gm_selection.UNAVAILABLE_RECORDS`, empty today,
so it normally drops nothing); its list is recorded in every manifest (`db_exclusions`). The
database changed on 2026-10-29, so every stripe of this set reads as stale and the next run
reselects all of them, replacing the earlier selection (made from `ESM-NGAsub_combined_old.csv`).

**Upstream**:

| Input | Source |
|---|---|
| GCIM target distributions (complete set) | `gcim_dist_AvgSA_03.pickle` (nb `031`) |
| Combined selection GM database | `cfg["proc_data"]["gm_database"]`, filtered in memory (see above) |
| IML-based disagg shards + poe stats | `cfg["proc_data"]["AvgSA_03_disagg_data_shards"]`, `cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| GMM logic tree + correlation flatfiles | loaded inside `setup_AvgSA03_gcim_gm_selection()` |

**Downstream** — `033-gm_selection_AvgSA_03_stage2_postprocess.ipynb` reads the complete
`AvgSA_03_final_ensembles.pickle`, writes the per-stripe pickles + manifests for the stale ones, and
re-verifies provenance.

**Run order** — run top to bottom (after `031`), then `033`. First run over a fresh results folder
selects everything (~an hour); subsequent "added a few IMLs" runs select only the new stripes.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import numpy as np
import pandas as pd

from phd_project.config import config
from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    build_final_ensembles,
    find_stale_stripes,
)
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
    setup_AvgSA03_gcim_gm_selection,
    wanted_stripe_keys,
    SELECTION_CONFIG,
    stripe_source_fps,
    stripe_fingerprints,
    gm_db_for_selection,
)


cfg = load_config()
# SIDE QUEST (see side-quest_gm_selection_optimisation/README.md). This notebook belongs to
# ONE record set, pinned here - not the active study - so re-running it only ever touches
# that set's folders. GRID is the set's stripe grid from config.yaml: the LIVE grid since
# 2026-10-29 (the set is reselected on the rebuilt database). The frozen grid it was first
# selected on is kept in 06_gm_selection/ss/AvgSA_03__ks/imls_for_selection_AvgSA_03.json.
SS = "AvgSA_03__ks"
GRID = config.stripe_grid_path(cfg, SS)

C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\gm_selection.py:15: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
# canonical Stage-1 output (merged each run to stay the COMPLETE (site, iml) set)
final_ensembles_fp = config.ss_dir(cfg, "proc_data", "gm_selection", SS) / "AvgSA_03_final_ensembles.pickle"

# intermediate stage caches, one per round (provenance-guarded; per-batch scratch).
# Named {im}_rd{ii}_{selection|optimisation}.pickle.
GMS = config.ss_dir(cfg, "proc_data", "gm_selection", SS)
IM = "AvgSA_03"
n_rounds = len(SELECTION_CONFIG["round_unbounded"])
stage_fps = {
    "select":   [GMS / f"{IM}_rd{ii}_selection.pickle"    for ii in range(1, n_rounds + 1)],
    "optimise": [GMS / f"{IM}_rd{ii}_optimisation.pickle" for ii in range(1, n_rounds + 1)],
}

# Inputs used for provenance fingerprinting (hashed by their file bytes). These stage /
# final artifacts cover a whole batch, so the disagg entry is the digest of the ENTIRE
# shard set (disagg_shard_dir -> disagg_shards_digest), not any one site's shard — see
# gm_selection._disagg_fingerprint_input.
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"
source_fps = {
    "gm_db_file":        cfg["proc_data"]["gm_database"],
    "gcim_file":         gcim_dist_fp,
    "disagg_shard_dir":  cfg["proc_data"]["AvgSA_03_disagg_data_shards"],
    "disagg_stats_file": cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"],
    "site_model_file":   cfg["hazard_models"]["eshm20_wp1_site_model"],
}

# The selection scheme (rounds, bounds, shuffle, seeds, rng_seed) now lives in the
# shared SELECTION_CONFIG (setup_AvgSA03_gm_selection.py) so 031/032/033 fingerprint
# each stripe identically. See that constant for the round-by-round description.

# other stuff:
rng_seed = SELECTION_CONFIG["rng_seed"]

# Escape hatch: force-reselect ALL wanted stripes, ignoring the per-stripe manifests
# (and bypassing every stage's staleness guard). Leave False for normal incremental
# runs: only the stale/new (site, iml) are (re)selected.
FORCE_RECOMPUTE = False

In [4]:
# Set up the selection context WITHOUT loading any disagg data (sites=()) — the stale
# check needs only the context and the index-derived key set. The shards for the stale
# sites are loaded in the next compute cell, once we know which sites those are.
_, disagg_stats, site_model, basic_selection_ctx, gm_db_full = setup_AvgSA03_gcim_gm_selection(sites=(), imls_for_selection_fp=GRID)

# THE database filter (the only place it happens; shared with nb 032-optimised): drop
# every record in UNAVAILABLE_RECORDS -- all versions of the list -- in memory, keeping
# the original index labels. From here on ONLY `gm_db` (filtered) is used; `gm_db_full`
# is kept for the leak check at the end of the notebook.
gm_db = gm_db_for_selection(gm_db_full)
excluded_idx = gm_db_full.index.difference(gm_db.index)
# Dropping 0 rows is the NORMAL case since 2026-10-29: nb 030 removes the records known to
# be unusable from the database file itself (unavailable_records_ngasub.csv), and
# UNAVAILABLE_RECORDS is empty. It is only non-empty for a record found bad after a
# selection, when just the stripes containing it should be reselected.
print(f"{len(excluded_idx)} database rows excluded in memory (0 is normal, see above).")

# Incremental cache: (re)select only stripes that are missing or stale on disk. Uses
# the same per-stripe fingerprint as 031/033 (excludes the IML subset JSON), so this
# matches the batch nb 031 just built gcim for. The fingerprint is stripe-precise:
# it hashes this (site, iml)'s own disagg DataFrame and its own stats row, not the
# whole shard or the whole stats pickle.
#
# It is also versioned by exclusion list (stripe_fingerprints): a stripe selected under
# an older version of UNAVAILABLE_RECORDS -- including the stripes selected before the
# list existed, whose manifests carry no db_exclusions entry -- stays valid as long as
# it contains none of the records excluded since. Only a stripe that contains a newly
# excluded record is reselected.
RESULT_FOLDER = config.ss_dir(cfg, "results", "record_selection", SS)
source_fps_stripe = stripe_source_fps()
wanted = wanted_stripe_keys(imls_for_selection_fp=GRID)
fp_fn, superseded = stripe_fingerprints(
    SELECTION_CONFIG, source_fps_stripe, basic_selection_ctx, disagg_stats)
to_compute, valid = find_stale_stripes(wanted, RESULT_FOLDER, fp_fn, superseded=superseded)
print(f"{len(wanted)} wanted (site, iml): {len(valid)} already valid, "
      f"{len(to_compute)} to (re)select.")

# load the (complete) gcim distributions written by nb 031
if gcim_dist_fp.is_file():
    with open(gcim_dist_fp, "rb") as file:
        gcim_dists = pickle.load(file)
    print(f"Loaded gcim for {len(gcim_dists)} (site, iml).")
else:
    gcim_dists = {}
    print("No GCIM distribution data found — run nb 031 first.")

Built 0 (site, iml) disaggregations across 0 sites (0 of 60 shards loaded; 563 (site, iml) wanted in total).
gm_db: 128760 -> 128760 rows (0 dropped)
0 database rows excluded in memory (0 is normal, see above).
563 wanted (site, iml): 0 already valid, 563 to (re)select.
Loaded gcim for 563 (site, iml).


## Stage 1 - Build final ensembles (slow compute)

Runs the configurable multi-round selection + optimisation engine and saves the canonical
`AvgSA_03_final_ensembles.pickle` (+ a `.manifest.json` provenance sidecar). Each round
drops progressively more causal-parameter bounds (`round_unbounded`), and each step is
provenance-cached: if an input (gm_db, gcim distributions, disagg, site model, params,
rng_seed, pickagm version, round config) changed since the cache was written, the stage is
recomputed instead of silently reusing stale data - set `FORCE_RECOMPUTE = True` to rebuild.

Rounds 1–3 (`force_optimisation = [False, True, False]`) reproduce the legacy AvgSA_03 result
exactly. **Round 4** re-optimises any `(site, poe)` still failing after round 3 over
`n_shuffles = 5` shuffled database orderings and keeps the best-scoring ensemble per
`(site, poe)` (replaced only if it passes or scores strictly better). Each round prints a
`[bounded: … | unbounded: …]` bounds label plus a selection-phase and optimisation-phase
breakdown; cached stages print `[cache] ... loaded`.

Post-processing (result pickles, plots, download/convert CSVs) lives in the separate,
fast notebook **`033-gm_selection_AvgSA_03_stage2_postprocess.ipynb`**.

In [5]:
# Force override: reselect every wanted stripe (ignores the per-stripe manifests).
if FORCE_RECOMPUTE:
    to_compute = list(wanted)

# Capture the prior COMPLETE final_ensembles before build_final_ensembles overwrites
# the file with just this batch. On a forced full rebuild we start clean.
prior_final = {}
if final_ensembles_fp.is_file() and not FORCE_RECOMPUTE:
    with open(final_ensembles_fp, "rb") as f:
        prior_final = pickle.load(f)

# Subset to the stale batch and run the multi-round engine over just those keys. The
# round loop is per-(site, iml) independent, so selecting a subset is equivalent to
# selecting them within the full set. gcim for every batch key must be present (built
# by nb 031); a missing key means 031 was not run for this batch.
#
# The disagg shards are loaded HERE, and only for the sites with stale stripes — an
# all-valid run reads no disagg data at all. The `gm_db` this call returns is the
# UNFILTERED database and is deliberately discarded (`*_`): only the filtered `gm_db`
# from the set-up cell is ever passed on.
if to_compute:
    stale_sites = {s for s, _ in to_compute}
    site_iml_disaggs, *_ = setup_AvgSA03_gcim_gm_selection(sites=stale_sites, imls_for_selection_fp=GRID)
    batch = {k: site_iml_disaggs[k] for k in to_compute}
else:
    batch = {}

missing = [k for k in batch if k not in gcim_dists]
if missing:
    raise KeyError(
        f"{len(missing)} stale (site, iml) have no gcim — run nb 031 first. "
        f"e.g. {missing[:5]}")

if batch:
    batch_final = build_final_ensembles(
        batch,
        disagg_stats,
        gcim_dists,
        gm_db,                      # FILTERED database (set-up cell)
        basic_selection_ctx,
        site_model,
        source_fps=source_fps,
        stage_fps=stage_fps,
        output_fp=final_ensembles_fp,
        round_unbounded=SELECTION_CONFIG["round_unbounded"],
        force_optimisation=SELECTION_CONFIG["force_optimisation"],
        shuffle=SELECTION_CONFIG["shuffle"],
        n_shuffles=SELECTION_CONFIG["n_shuffles"],
        shuffle_rng_seeds=SELECTION_CONFIG["shuffle_rng_seeds"],
        rng_seed=rng_seed,
        force_recompute=FORCE_RECOMPUTE,
        # records the in-memory filter in the stage + final manifests (does not filter)
        db_exclusions=SELECTION_CONFIG["db_exclusions"],
    )
else:
    batch_final = {}
    print("All stripes valid — nothing to (re)select.")

# MERGE the batch into the prior complete set so final_ensembles.pickle stays the
# COMPLETE (site, iml) selection: it is consumed as complete by
# summarise_record_availability.py and provenance-stamped by nb 040. Prune any key no
# longer wanted (an IML removed from a union list). The manifest build_final_ensembles
# wrote is source-file based, so it stays valid for the merged file.
wanted_set = set(wanted)
final_ensembles = {k: v for k, v in {**prior_final, **batch_final}.items() if k in wanted_set}
if batch_final or prior_final:
    with open(final_ensembles_fp, "wb") as f:
        pickle.dump(final_ensembles, f)
print(f"final_ensembles.pickle now holds {len(final_ensembles)} (site, iml) "
      f"({len(batch_final)} (re)selected this run).")

Built 563 (site, iml) disaggregations across 60 sites (60 of 60 shards loaded; 563 (site, iml) wanted in total).

================ Round 1/4 ================
Bounds  [bounded: m, d, vs30 | unbounded: none]
Selection phase — all 563 (site, iml) [first round]
AvgSA_03_rd1_selection.pickle (C:\Users\clemettn\Documents\phd\data_processed\06_gm_selection\ss\AvgSA_03__ks)


R1/4 select (563 sites):   0%|          | 0/563 [00:00<?, ?it/s]

    -> 312 pass, 251 fail KS  (of which 9 could not form a record set at all)
Optimisation phase — 251 (site, iml)  [only sites still failing]
    (9 of these still have no record set -> skipped by the optimiser)
AvgSA_03_rd1_optimisation.pickle (C:\Users\clemettn\Documents\phd\data_processed\06_gm_selection\ss\AvgSA_03__ks)


R1/4 optimise (251 sites):   0%|          | 0/251 [00:00<?, ?it/s]

Optimizing Ensemble Slots:   0%|          | 0/30 [00:00<?, ?it/s]

Optimizing Ensemble Slots:   0%|          | 0/30 [00:00<?, ?it/s]

... (344 outputs left out)

Optimizing Ensemble Slots:   0%|          | 0/30 [00:00<?, ?it/s]

Shuffle 5 - Optimising Ensemble:   0%|          | 0/1 [00:00<?, ?it/s]

Optimizing Ensemble Slots:   0%|          | 0/30 [00:00<?, ?it/s]

    -> 0 now pass, 1 still fail
      (keep-best: replaced 0, kept incumbent for 1)
-> 1 (site, iml) carried into the next round

Sub-Optimal! - 1 (site, iml) still without a passing ensemble after 4 rounds
final_ensembles.pickle now holds 563 (site, iml) (563 (re)selected this run).


In [6]:
# Isolate the (site, iml) that still fail the KS test after all rounds (across the full
# merged set), mapping each to its list of failing IMs. Not pickled - just a quick look.
failing_ensembles = {
    k: v["ks_failed_ims"]
    for k, v in final_ensembles.items()
    if v is not None and not v["ks_passed"]
}

print(f"{len(failing_ensembles)} (site, iml) still failing:")
for k in failing_ensembles:
    print("  ", k, "->", failing_ensembles[k])

1 (site, iml) still failing:
   (49, 1.15) -> ['PGA']


## Check: no excluded record was selected

Every selected record is identified by its row label in the database. Since the filter keeps
the original labels, an excluded record would show up here as one of the dropped labels.

In [7]:
# Look for any dropped row label in every ensemble of the (merged, complete) set.
leaks = {
    k: sorted(set(v["recs"].index).intersection(excluded_idx))
    for k, v in final_ensembles.items()
    if v is not None
}
leaks = {k: v for k, v in leaks.items() if v}
if leaks:
    raise RuntimeError(f"Excluded records were selected in {len(leaks)} stripe(s): {leaks}")
print(f"OK - none of the {len(excluded_idx)} database rows excluded in memory appears in any "
      f"of the {sum(v is not None for v in final_ensembles.values())} ensembles (the records "
      f"removed by nb 030 are not in the database at all).")

OK - none of the 0 database rows excluded in memory appears in any of the 563 ensembles (the records removed by nb 030 are not in the database at all).
